# CS 598 MP3 - DeepSeekCoder experiments

This notebook generates the MP3 datasets, runs the vanilla and crafted experiments for both tasks, evaluates every response, and packages the deliverables.

Before running: commit and push the current MP3 implementation to the repository, open this notebook in Google Colab, and select **Runtime > Change runtime type > T4 GPU**. Run the cells from top to bottom.

In [ ]:
# Confirm that this runtime has the required GPU before doing any setup.
import os
import subprocess
import sys

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No NVIDIA GPU detected. Select Runtime > Change runtime type > T4 GPU, then restart.')
print(gpu.stdout)

## 1. Clone the team repository

The cell is rerunnable: an existing clean checkout is updated with a fast-forward pull.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ramprakashreddy1/CS598JBR-Team-16.git'
BRANCH = 'main'
REPO_DIR = Path('/content/CS598JBR-Team-16')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

MP3_DIR = REPO_DIR / 'MP3'
os.chdir(MP3_DIR)
print('Working directory:', Path.cwd())

### Upload local development overrides

Use this cell when the MP3 development changes have not been pushed to GitHub. Select `task_1.py`, `task_2.py`, `analyze.py`, and `setup_dataset.sh` from your local `MP3` directory. The uploaded files replace only the corresponding files in the disposable Colab checkout.

In [ ]:
from google.colab import files
import re

uploaded = files.upload()
required_overrides = {'task_1.py', 'task_2.py', 'analyze.py'}
normalized_uploads = set()
for uploaded_name, contents in uploaded.items():
    # Colab renames an upload when the cloned destination already exists,
    # for example task_1.py -> task_1 (2).py. Restore the intended name.
    canonical_name = re.sub(r' \(\d+\)(?=\.[^.]+$)', '', uploaded_name)
    if canonical_name in required_overrides | {'setup_dataset.sh', 'setup_models.sh'}:
        Path(canonical_name).write_bytes(contents)
        normalized_uploads.add(canonical_name)
        if uploaded_name != canonical_name:
            Path(uploaded_name).unlink(missing_ok=True)

missing_overrides = required_overrides - normalized_uploads
if missing_overrides:
    raise AssertionError(f'Upload the missing files and rerun this cell: {sorted(missing_overrides)}')
print('Installed overrides:', sorted(normalized_uploads))
print('Destination:', Path.cwd())

## 2. Install dependencies and verify Java

Task 1 compiles untrusted model output. This notebook runs it only inside the disposable Colab runtime.

In [ ]:
%%bash
set -euo pipefail
# Update starter pins for current Colab (Python 3.13 and CUDA 13).
sed -i 's/bitsandbytes==0.47.0/bitsandbytes==0.50.2/' setup_models.sh
sed -i 's/sentencepiece==0.1.99/sentencepiece==0.2.2/' setup_models.sh
bash setup_dataset.sh
bash setup_models.sh
if ! command -v javac >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq default-jdk-headless
fi
javac -version
java -version

In [ ]:
# Experiment configuration. Set RESUME_EXISTING_RESULTS=True only when continuing an interrupted run.
SEED = '148118091928943694313340081131652963687'
MODEL = 'deepseek-ai/deepseek-coder-6.7b-instruct'
RESUME_EXISTING_RESULTS = False
os.environ['MP3_RESUME'] = '1' if RESUME_EXISTING_RESULTS else '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('Seed:', SEED)
print('Model:', MODEL)
print('Resume:', RESUME_EXISTING_RESULTS)

In [ ]:
# Stream a command to the notebook and save the exact same output as its required log.
def run_logged(command, log_name):
    print('$', ' '.join(command))
    with open(log_name, 'w', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def count_jsonl(path):
    import json
    with open(path, encoding='utf-8') as file:
        rows = [json.loads(line) for line in file if line.strip()]
    return len(rows), rows


## 3. Copy the MP1 selection and generate HumanEval-X

In [ ]:
import shutil

selected_name = f'selected_humaneval_{SEED}.jsonl'
source_dataset = REPO_DIR / 'MP1' / selected_name
selected_dataset = MP3_DIR / selected_name
if not source_dataset.exists():
    raise FileNotFoundError(source_dataset)
shutil.copy2(source_dataset, selected_dataset)
print('Copied:', selected_dataset)

run_logged([sys.executable, 'humanevalx_dataset_generation.py', selected_name], 'humanevalx_dataset_generation.log')

In [ ]:
python_dataset = f'selected_humanevalx_python_{SEED}.jsonl'
java_dataset = f'selected_humanevalx_java_{SEED}.jsonl'

python_count, python_rows = count_jsonl(python_dataset)
java_count, java_rows = count_jsonl(java_dataset)
assert python_count == java_count == 20
assert {row['task_id'] for row in python_rows} == {row['task_id'] for row in java_rows}
print('HumanEval-X Python and Java datasets contain the same 20 task IDs.')

## 4. Task 1 - Python-to-Java translation

Each command loads the model, generates 20 responses, compiles the Java, runs the supplied tests, and checkpoints the JSONL after every problem. If a cell is interrupted, set `RESUME_EXISTING_RESULTS=True`, rerun the configuration cell, and rerun that experiment cell.

In [ ]:
task_1_vanilla = f'task_1_{SEED}_vanilla.jsonl'
if not RESUME_EXISTING_RESULTS:
    Path(task_1_vanilla).unlink(missing_ok=True)
run_logged([sys.executable, 'task_1.py', python_dataset, MODEL, task_1_vanilla, 'True'], 'task_1_vanilla.log')

In [ ]:
task_1_crafted = f'task_1_{SEED}_crafted.jsonl'
if not RESUME_EXISTING_RESULTS:
    Path(task_1_crafted).unlink(missing_ok=True)
run_logged([sys.executable, 'task_1.py', python_dataset, MODEL, task_1_crafted, 'False'], 'task_1_crafted.log')

## 5. Generate HumanEvalPack and run Task 2

In [ ]:
run_logged([sys.executable, 'humanevalpack_dataset_generation.py', selected_name], 'humanevalpack_dataset_generation.log')
humanevalpack_dataset = f'selected_humanevalpack_{SEED}.jsonl'
pack_count, pack_rows = count_jsonl(humanevalpack_dataset)
assert pack_count == 20
assert {row['task_id'] for row in pack_rows} == {row['task_id'] for row in python_rows}
print('HumanEvalPack dataset contains the expected 20 task IDs.')

In [ ]:
task_2_vanilla = f'task_2_{SEED}_vanilla.jsonl'
if not RESUME_EXISTING_RESULTS:
    Path(task_2_vanilla).unlink(missing_ok=True)
run_logged([sys.executable, 'task_2.py', humanevalpack_dataset, MODEL, task_2_vanilla, 'True'], 'task_2_vanilla.log')

In [ ]:
task_2_crafted = f'task_2_{SEED}_crafted.jsonl'
if not RESUME_EXISTING_RESULTS:
    Path(task_2_crafted).unlink(missing_ok=True)
run_logged([sys.executable, 'task_2.py', humanevalpack_dataset, MODEL, task_2_crafted, 'False'], 'task_2_crafted.log')

## 6. Analyze and verify deliverables

In [ ]:
run_logged([sys.executable, 'analyze.py', SEED, 'analysis.md'], 'analysis.log')

result_files = [task_1_vanilla, task_1_crafted, task_2_vanilla, task_2_crafted]
for path in result_files:
    count, rows = count_jsonl(path)
    assert count == 20, f'{path}: expected 20 rows, found {count}'
    assert all(isinstance(row.get('is_correct'), bool) for row in rows), f'{path}: invalid is_correct value'
    passed = sum(row['is_correct'] for row in rows)
    print(f'{path}: {passed}/20 correct ({passed * 5:.1f}%)')

print('\n--- analysis.md ---\n')
print(Path('analysis.md').read_text())

In [ ]:
# Verify every file required by the supplied validator exists locally.
required = [
    selected_name, humanevalpack_dataset, python_dataset, java_dataset,
    task_1_vanilla, task_1_crafted, task_2_vanilla, task_2_crafted,
    'task_1.py', 'task_2.py',
    'humanevalx_dataset_generation.log', 'humanevalpack_dataset_generation.log',
    'task_1_vanilla.log', 'task_1_crafted.log',
    'task_2_vanilla.log', 'task_2_crafted.log',
]
missing = [path for path in required if not Path(path).exists()]
if missing:
    raise AssertionError(f'Missing deliverables: {missing}')
print(f'All {len(required)} required MP3 files are present.')

## 7. Package results

The archive is a backup for downloading from Colab. The required files must still be committed to the repository's `main` branch, and the progress-report PDF must be submitted through Canvas.

In [ ]:
import zipfile

archive = Path(f'/content/MP3_deliverables_{SEED}.zip')
package_files = required + ['analysis.md', 'analysis.log']
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in package_files:
        bundle.write(path, arcname=f'MP3/{path}')
print('Created:', archive, f'({archive.stat().st_size / 1024 / 1024:.2f} MiB)')

# Uncomment these two lines to download the archive to your computer.
# from google.colab import files
# files.download(str(archive))